# Group Assignment: Pokemon Combat Simulator Dashboard

## Overview

In this group assignment you'll build a **Pokemon combat simulator** as an interactive Streamlit dashboard. Your app will fetch real Pokemon data from the PokeAPI, display it in a polished layout, and simulate turn-based battles between two Pokemon.

This ties together everything you've learned in PDA2: **APIs** (sessions 7-8), **pandas** (throughout), **OOP** (sessions 9-12), and **Streamlit** (session 14).

Unlike previous homework, there is **no skeleton file** — your group designs and builds the app from scratch.

## Estimated Time: 10-15 hours (shared across the group)

## Deliverable

Submit a **single link** to your public GitHub repository. The repo must contain:

| Required in repo | Description |
|------------------|-------------|
| **Streamlit app** | Your main `.py` file (e.g., `dashboard.py`) |
| **`requirements.txt`** | Python dependencies (in the repo root) |
| **`README.md`** | Must include: (1) a **link to the deployed Streamlit app**, and (2) a **contributions section** listing each member's contributions |
| **`.streamlit/config.toml`** | Theme configuration (optional but recommended) |

---

## The PokeAPI

You'll use the [PokeAPI](https://pokeapi.co/) to fetch all Pokemon data. It's free, requires **no authentication**, and has **no rate limiting**.

**Base URL:** `https://pokeapi.co/api/v2/`

You'll need **three endpoints**:

### 1. `GET /pokemon/{name}` — Pokemon data

Returns stats, types, moves list, and sprites.

```python
import requests

response = requests.get("https://pokeapi.co/api/v2/pokemon/pikachu")
data = response.json()

# Name
data["name"]  # "pikachu"

# Sprite (image URL)
data["sprites"]["front_default"]
# "https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/25.png"

# Types (a Pokemon can have 1 or 2 types)
types = [t["type"]["name"] for t in data["types"]]
# ["electric"]

# Base stats
stats = {s["stat"]["name"]: s["base_stat"] for s in data["stats"]}
# {"hp": 35, "attack": 55, "defense": 40,
#  "special-attack": 50, "special-defense": 50, "speed": 90}

# Move names (these are attacks, not abilities)
move_names = [m["move"]["name"] for m in data["moves"]]
# ["mega-punch", "pay-day", "thunder-punch", ...]
```

> **Note:** "moves" are attacks a Pokemon can use in battle (Thunderbolt, Flamethrower, etc.). Don't confuse them with "abilities", which are passive traits.

### 2. `GET /move/{name}` — Move details

Returns power, accuracy, type, and damage class for a specific move.

```python
response = requests.get("https://pokeapi.co/api/v2/move/thunderbolt")
move = response.json()

move["power"]                     # 90
move["accuracy"]                  # 100
move["type"]["name"]              # "electric"
move["damage_class"]["name"]      # "special" (or "physical")
```

- `damage_class` tells you whether the move uses Attack/Defense (`"physical"`) or Special-Attack/Special-Defense (`"special"`).
- Some moves have `power: None` (status moves like Growl) — you'll filter these out.

### 3. `GET /type/{name}` — Type effectiveness

Returns damage relations showing which types a given move type is strong/weak against.

```python
response = requests.get("https://pokeapi.co/api/v2/type/electric")
type_data = response.json()

dr = type_data["damage_relations"]

# Electric does 2x damage to:
[t["name"] for t in dr["double_damage_to"]]   # ["flying", "water"]

# Electric does 0.5x damage to:
[t["name"] for t in dr["half_damage_to"]]      # ["grass", "electric", "dragon"]

# Electric does 0x damage to:
[t["name"] for t in dr["no_damage_to"]]        # ["ground"]
```

Use these relations to compute the type effectiveness multiplier for each attack.

---

## Dashboard Requirements

Your dashboard must include **6 sections**:

### 1. Pokemon Selection

Let the user pick two Pokemon to battle. You can use `st.selectbox`, `st.text_input`, or any other input widget.

### 2. Pokemon Display

For each selected Pokemon, show:
- The Pokemon's **sprite** (image from `sprites.front_default`)
- **Name** and **types**
- **Base stats** (hp, attack, defense, special-attack, special-defense, speed)

### 3. Move Selection

For each Pokemon, let the user pick **one damaging move** (a move with `power > 0`) from that Pokemon's move list. Display the selected move's power, accuracy, type, and damage class.

### 4. Stat Comparison Chart

A **Plotly grouped bar chart** comparing both Pokemon's base stats side by side.

### 5. Combat Simulation

A **"Battle!" button** that runs the turn-based combat simulation (see Combat Mechanics below). Display:
- A **battle log table** showing what happened each round
- A **winner announcement**

### 6. HP Over Time Chart

A **Plotly line chart** showing both Pokemon's HP decreasing over the rounds of battle.

---

## Combat Mechanics

Use the following simplified damage formula (based on the real Pokemon games):

```python
import random

level = 50  # fixed for all Pokemon

# Choose attack/defense stats based on the move's damage class
if move_damage_class == "physical":
    attack_stat = attacker_stats["attack"]
    defense_stat = defender_stats["defense"]
else:  # "special"
    attack_stat = attacker_stats["special-attack"]
    defense_stat = defender_stats["special-defense"]

# Type effectiveness multiplier
# Check the move's type against EACH of the defender's types
# Multiply together (e.g., 2.0 * 0.5 = 1.0 for a dual-type defender)
effectiveness = 1.0
for defender_type in defender_types:
    if defender_type in double_damage_to:
        effectiveness *= 2.0
    elif defender_type in half_damage_to:
        effectiveness *= 0.5
    elif defender_type in no_damage_to:
        effectiveness *= 0.0

# Accuracy check — miss means 0 damage
if random.random() < (move_accuracy / 100):
    damage = int(
        ((2 * level / 5 + 2) * move_power * (attack_stat / defense_stat) / 50 + 2)
        * effectiveness
    )
else:
    damage = 0  # miss!
```

### Battle Rules

- Each Pokemon selects **one move before the battle starts**. That same move is used **every round** until the battle ends.
- **Speed** determines turn order: the faster Pokemon attacks first each round. If speeds are equal, randomize who goes first.
- Each round, both Pokemon attack once (unless one faints mid-round).
- The battle ends when one Pokemon's **HP drops to 0 or below**.
- **Cap at 100 rounds** to prevent infinite loops (declare a draw if reached).

---

## Pandas Requirements

Your dashboard must use pandas DataFrames in **three specific places**:

### 1. Stat Comparison DataFrame

Build a DataFrame from the API data and reshape it with `.melt()` for the grouped bar chart:

```python
import pandas as pd

# Build a DataFrame with one row per Pokemon
stat_df = pd.DataFrame([
    {"pokemon": "pikachu", "hp": 35, "attack": 55, "defense": 40,
     "special-attack": 50, "special-defense": 50, "speed": 90},
    {"pokemon": "charizard", "hp": 78, "attack": 84, "defense": 78,
     "special-attack": 109, "special-defense": 85, "speed": 100},
])

# Melt into tidy format for Plotly
melted = stat_df.melt(id_vars="pokemon", var_name="stat", value_name="value")
```

### 2. Battle Log DataFrame

Collect round-by-round results as a list of dicts, then convert:

```python
battle_log = [
    {"round": 1, "attacker": "pikachu", "move": "thunderbolt",
     "damage": 42, "defender_hp": 36},
    {"round": 1, "attacker": "charizard", "move": "flamethrower",
     "damage": 58, "defender_hp": 0},
]

log_df = pd.DataFrame(battle_log)
```

### 3. HP Over Time DataFrame

Track HP after each round in tidy format for the line chart:

```python
hp_history = [
    {"round": 0, "pokemon": "pikachu", "hp": 35},
    {"round": 0, "pokemon": "charizard", "hp": 78},
    {"round": 1, "pokemon": "pikachu", "hp": 0},
    {"round": 1, "pokemon": "charizard", "hp": 36},
]

hp_df = pd.DataFrame(hp_history)
```

---

## Tips & Hints

### Caching

Use `@st.cache_data` on **all API fetch functions** to avoid redundant calls. Each time the user changes a selection, Streamlit reruns the entire script — caching ensures you don't re-fetch data you already have.

```python
@st.cache_data
def fetch_pokemon(name):
    response = requests.get(f"https://pokeapi.co/api/v2/pokemon/{name}")
    return response.json()
```

> **Hashability note:** `@st.cache_data` needs all arguments to be hashable. If you pass a list of types, convert it to a **tuple** first (lists are not hashable, tuples are).

### Layout

Use `st.columns(2)` to display both Pokemon side by side:

```python
col1, col2 = st.columns(2)

with col1:
    st.image(sprite_url_1)
    st.write("Pikachu")

with col2:
    st.image(sprite_url_2)
    st.write("Charizard")
```

### Charts

Use `st.plotly_chart(fig, use_container_width=True)` to display Plotly charts.

### Edge Cases to Handle

- **Moves with `power: None`** — these are status moves (like Growl). Filter them out so users can only select damaging moves.
- **API errors** — show a friendly error message if a Pokemon name is invalid.
- **Same Pokemon selected twice** — this is fine to allow, but you might want to show a warning.

---

## Deployment Instructions

### 1. Create a Public GitHub Repository

Your repo should contain at minimum:

```
your-repo/
├── dashboard.py              # Your Streamlit app (name it whatever you like)
├── requirements.txt          # Python dependencies
├── README.md                 # App link + contributions
└── .streamlit/
    └── config.toml           # Theme configuration (optional)
```

A starter `requirements.txt` and `.streamlit/config.toml` are provided in the `deploy/` folder of this assignment.

### 2. Deploy to Streamlit Community Cloud

1. Go to [share.streamlit.io](https://share.streamlit.io)
2. Sign in with GitHub
3. Click **"New app"**
4. Select your repo, branch, and main file path
5. Click **"Deploy"**

### 3. Update Your README

Your `README.md` must include:
- A **link to the deployed app** (e.g., `https://your-app.streamlit.app`)
- A **contributions section** listing what each group member worked on

### 4. Submit

Submit a **single link** to your public GitHub repository. Everything will be graded from the repo.